<a href="https://colab.research.google.com/github/AaritShrama/Telco-Customer-Churn-Using-PyTorch/blob/main/Telco_Customer_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP-1 : IMPORT REQUIRED LIBRARIES

import pandas as pd
import matplotlib.pyplot  as plt
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import torch
from torch.utils.data import Dataset , DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
# STEP-2 : IMPORT DATAFRAME

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# STEP-3 : DROP ID COLUMN (IRRELEVANT TO THE NN)

df.drop('customerID' , axis = 1, inplace = True)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.dtypes

,0
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object
OnlineBackup,object


In [ ]:
# STEP-4 : FIX THE DATA IN THE TABLE TO MAKE IT APPROPRIATE FOR A NN

df['Churn']= df['Churn'].map({'Yes': 1, 'No': 0})

columns = ['gender', 'Partner', 'Dependents', 'PhoneService',
           'PaperlessBilling']
for column in columns:
  df[column] = df[column].map({'Yes': 1, 'No': 0,
                               'Male': 1, 'Female': 0})
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,1,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.5,0
2,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,0,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

y = df["Churn"]
X = df.drop("Churn", axis=1)

X = pd.get_dummies(X, drop_first=True)
X = X.astype(int)

/tmp/ipython-input-3143848126.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [ ]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29,29,1,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56,1889,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53,108,0,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42,1840,1,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70,151,0,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
y.head()

,Churn
0,0
1,0
2,1
3,0
4,1


In [ ]:
# STEP-5 : TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)


In [ ]:
# STEP-6: SCALING & TENSOR CONVERSION OF THE DATA

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train.values)
X_test  = scaler.transform(X_test.values)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test  = torch.tensor(y_test.values, dtype=torch.float32)


In [ ]:
# STEP-7 : CUSTOM DATASET CLASS FORMATION

class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
train_data = ChurnDataset(X_train, y_train)
test_data  = ChurnDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


In [ ]:
# STEP-8 : NEURAL NETWORK SETUP

class ChurnNN(nn.Module):

  def __init__(self , num_features):
    super().__init__()
    self.Network = nn.Sequential(
        nn.Linear(num_features , 32),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(32 , 16),
        nn.ReLU(),
        nn.Linear(16 , 1)
    )

  def forward(self,x):
    return self.Network(x)

model = ChurnNN(X_train.shape[1])



In [ ]:
# STEP-9 : PARAMETERS

loss = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 100

In [ ]:
# STEP-10 : TRAINING LOOP

for epoch in range (epochs):

  epoch_loss = 0.0

  for batch_features,batch_labels in train_loader:

    y_pred = model.forward(batch_features).squeeze()
    batch_loss = loss(y_pred, batch_labels)

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

    epoch_loss += batch_loss.item()

  avg_epoch_loss = epoch_loss / len(train_loader)
  print('epoch - ', epoch+1 , 'loss - ', avg_epoch_loss)


epoch -  1 loss -  0.4498397762660926
epoch -  2 loss -  0.42914655526815837
epoch -  3 loss -  0.42377174750896496
epoch -  4 loss -  0.42009507363798926
epoch -  5 loss -  0.41957251340319207
epoch -  6 loss -  0.41895948863972376
epoch -  7 loss -  0.41617062940435895
epoch -  8 loss -  0.4172130627820721
epoch -  9 loss -  0.4167425385807867
epoch -  10 loss -  0.40888130589056826
epoch -  11 loss -  0.40888033066428986
epoch -  12 loss -  0.4121997055865951
epoch -  13 loss -  0.41527185096579083
epoch -  14 loss -  0.4065217015433446
epoch -  15 loss -  0.4070992860416908
epoch -  16 loss -  0.409253696095472
epoch -  17 loss -  0.4139354084171144
epoch -  18 loss -  0.41059991439520305
epoch -  19 loss -  0.4086191098568803
epoch -  20 loss -  0.40707099471388564
epoch -  21 loss -  0.4066905271535539
epoch -  22 loss -  0.40228856990566364
epoch -  23 loss -  0.40341995917471113
epoch -  24 loss -  0.40520606463575093
epoch -  25 loss -  0.40151634635561606
epoch -  26 loss -  

In [ ]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        # forward pass
        out = model(batch_features).squeeze()

        # sigmoid + threshold
        probs = torch.sigmoid(out)
        preds = (probs >= 0.5).float()

        total += batch_labels.size(0)
        correct += (preds == batch_labels).sum().item()

print("Accuracy -", correct / total)


Accuracy - 0.7821149751596878
